# Análisis de emisiones de CO₂ en Colombia
**Daniel Carrillo y Daniela Baquero** — Actividad 4, Ciencia de Datos.

Fuente: Our World in Data, indicador de emisiones anuales de CO₂. Este primer avance se centra en descarga, inspección y limpieza.


In [23]:
from google.colab import files

files.download("datos/co2_colombia_limpio.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download("datos/co2_colombia_limpio.csv")

In [12]:
import pandas as pd

url = 'https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv'

df = pd.read_csv(url)

colombia = df[df['iso_code'] == 'COL'].copy()

print('Registros de Colombia:', len(colombia))
print(colombia[['country', 'year', 'co2']].tail())

Registros de Colombia: 276
        country  year      co2
11028  Colombia  2021  103.415
11029  Colombia  2022  101.544
11030  Colombia  2023  105.349
11031  Colombia  2024  106.288
11032  Colombia  2025   96.067


In [20]:

colombia_limpio = colombia[["country", "year", "co2"]].copy()

colombia_limpio = colombia_limpio.dropna().drop_duplicates()
colombia_limpio = colombia_limpio[colombia_limpio["co2"] >= 0]
colombia_limpio = colombia_limpio.sort_values("year")

print("Datos limpios:", len(colombia_limpio))

Datos limpios: 170


In [ ]:
import pandas as pd

# Seleccionar las columnas necesarias
colombia_limpio = colombia[['country', 'year', 'co2']].copy()

# Revisar valores faltantes y duplicados
print('Valores faltantes antes de limpiar:')
print(colombia_limpio.isnull().sum())
print('Duplicados:', colombia_limpio.duplicated().sum())

# Eliminar registros incompletos, duplicados y emisiones negativas
colombia_limpio = colombia_limpio.dropna().drop_duplicates()
colombia_limpio = colombia_limpio[colombia_limpio['co2'] >= 0]
colombia_limpio = colombia_limpio.sort_values('year')

print('\nDatos después de limpiar:', len(colombia_limpio))
print(colombia_limpio.describe())
colombia_limpio.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(colombia_limpio['year'], colombia_limpio['co2'], color='royalblue', linewidth=2)
plt.title('Evolución histórica de las emisiones de CO₂ en Colombia')
plt.xlabel('Año')
plt.ylabel('Emisiones de CO₂ (millones de toneladas)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(
    colombia_limpio["year"],
    colombia_limpio["co2"],
    color="royalblue",
    linewidth=2
)

plt.title("Evolución histórica de las emisiones de CO₂ en Colombia")
plt.xlabel("Año")
plt.ylabel("Emisiones de CO₂ (millones de toneladas)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Interpretación de la evolución histórica del CO₂ en Colombia

La visualización evidencia una tendencia general creciente de las emisiones anuales de dióxido de carbono en Colombia. Durante las primeras décadas de la serie histórica, los valores permanecieron relativamente bajos. A partir de mediados del siglo XX se observa un crecimiento sostenido, que se intensifica en las décadas posteriores.

En los años más recientes se identifican fluctuaciones importantes, con algunos valores superiores a 100 millones de toneladas anuales. Estas variaciones indican que el crecimiento de las emisiones no ha sido uniforme.

La gráfica permite describir la evolución histórica, pero por sí sola no demuestra cuáles fueron las causas económicas, energéticas o sociales de los cambios observados.

Estos resultados justifican continuar con un análisis estadístico y un modelo de regresión lineal para evaluar la relación entre el tiempo y las emisiones de CO₂.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Seleccionar el período de análisis
datos_modelo = colombia_limpio[
    colombia_limpio["year"] >= 1950
].sort_values("year").copy()

# Separar entrenamiento y prueba cronológicamente
corte = int(len(datos_modelo) * 0.8)
entrenamiento = datos_modelo.iloc[:corte]
prueba = datos_modelo.iloc[corte:]

X_train = entrenamiento[["year"]]
y_train = entrenamiento["co2"]
X_test = prueba[["year"]]
y_test = prueba["co2"]

# Entrenar el modelo
modelo = LinearRegression()
modelo.fit(X_train, y_train)

# Realizar predicciones
predicciones = modelo.predict(X_test)

# Evaluar el modelo
print("Pendiente:", round(modelo.coef_[0], 3))
print("MAE:", round(mean_absolute_error(y_test, predicciones), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, predicciones)), 3))
print("R²:", round(r2_score(y_test, predicciones), 3))

# Crear gráfica
plt.figure(figsize=(12, 6))
plt.scatter(X_train["year"], y_train, label="Entrenamiento")
plt.scatter(X_test["year"], y_test, label="Datos reales")
plt.plot(
    X_test["year"],
    predicciones,
    color="red",
    linewidth=2,
    label="Regresión lineal"
)

plt.title("Regresión lineal de emisiones de CO₂ en Colombia")
plt.xlabel("Año")
plt.ylabel("Emisiones de CO₂ (millones de toneladas)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Interpretación de los resultados de la regresión lineal

Se implementó un modelo de regresión lineal simple para analizar la relación entre el año y las emisiones anuales de dióxido de carbono (CO₂) en Colombia, utilizando información histórica desde 1950.

Los datos se dividieron cronológicamente en un conjunto de entrenamiento correspondiente al 80 % de las observaciones y un conjunto de prueba equivalente al 20 % restante.

El modelo obtuvo un error absoluto medio (MAE) de 10,861 millones de toneladas y una raíz del error cuadrático medio (RMSE) de 12,080 millones de toneladas.

Por otra parte, el coeficiente de determinación (R²) fue de -0,441, lo que indica que el modelo presenta un desempeño predictivo limitado sobre los datos de prueba.

Aunque la regresión lineal permite representar una tendencia histórica creciente, no logra reproducir adecuadamente las fluctuaciones observadas durante los años recientes.

Estos resultados evidencian que las emisiones de CO₂ no siguen necesariamente un comportamiento lineal constante. Para mejorar las predicciones podrían incorporarse variables adicionales, como el consumo energético, la actividad industrial y otros indicadores económicos.

En conclusión, el modelo resulta útil para comprender la tendencia general de las emisiones, pero sus resultados no deben interpretarse como predicciones precisas del comportamiento futuro del CO₂ en Colombia.

## Selección de Colombia
Filtramos por código ISO COL y conservamos la serie temporal de emisiones.


In [ ]:
indicador = next(c for c in df.columns if c not in ["Entity", "Code", "Year"])
co = df.loc[df["Code"].eq("COL"), ["Entity", "Code", "Year", indicador]].copy()
co.columns = ["pais", "codigo", "anio", "co2_toneladas"]
co.head()


## Diagnóstico y limpieza
Revisamos nulos, duplicados, años repetidos y emisiones negativas. No imputamos datos faltantes sin justificación.


In [ ]:
print("Nulos:", co.isna().sum())
print("Duplicados:", co.duplicated().sum())
co["anio"] = pd.to_numeric(co["anio"], errors="coerce")
co["co2_toneladas"] = pd.to_numeric(co["co2_toneladas"], errors="coerce")
co = co.drop_duplicates().dropna(subset=["anio", "co2_toneladas"])
co = co[co["co2_toneladas"] >= 0].sort_values("anio").reset_index(drop=True)
assert not co["anio"].duplicated().any(), "Existen años repetidos"
co["anio"] = co["anio"].astype(int)
co["co2_millones_toneladas"] = co["co2_toneladas"] / 1e6
print("Años:", co.anio.min(), co.anio.max())
co.describe()


## Exportación
Ejecutar desde la raíz del proyecto para guardar los datos limpios.


In [22]:
from pathlib import Path
Path("datos").mkdir(exist_ok=True)
colombia_limpio.to_csv("datos/co2_colombia_limpio.csv", index=False)
print("CSV guardado")


CSV guardado


## Siguientes avances
1. Daniela: visualizaciones y análisis exploratorio.
2. Ambos: regresión lineal con evaluación temporal, interpretación y conclusiones.

Referencia: Ritchie y Roser, *CO₂ emissions*, Our World in Data, https://ourworldindata.org/co2-emissions.


In [ ]:

from pathlib import Path

# Crear carpeta para los datos procesados
Path("datos").mkdir(exist_ok=True)

# Guardar los datos limpios
colombia_limpio.to_csv(
    "datos/co2_colombia_limpio.csv",
    index=False
)

print("Archivo CSV generado correctamente")
print("Registros guardados:", len(colombia_limpio))
